In [36]:
import yfinance as yf
from fredapi import Fred
import pandas as pd

'''
FredAPI Key: 2bc6a527e0a892d7b9862b9a43034900
'''
fred = Fred(api_key="2bc6a527e0a892d7b9862b9a43034900")

Code	Meaning
d	Daily
w	Weekly
bw	Biweekly
m	Monthly
q	Quarterly
sa	Semiannual
a	Annual

You can also specify aggregation method when downsampling:
aggregation_method
avg
sum
eop (end of period)

fred.get_series()
Core Parameters
Parameter	Description
series_id	FRED series code (e.g., 'GDP')
observation_start	Start date (YYYY-MM-DD)
observation_end	End date
frequency	Convert to lower frequency
aggregation_method	avg, sum, eop
units	Transformations
realtime_start	For vintage data
realtime_end	For vintage data

You can request transformed data directly.

Code	Meaning
lin	Level (default)
chg	Change
ch1	Year-over-year change
pch	Percent change
pc1	Year-over-year percent change
log	Natural log

Format: yyyy-mm-dd

In [37]:
# data = yf.download("SYMBOL", start="YYYY-MM-DD", end="YYYY-MM-DD", interval="INTERVAL")

sp500 = yf.download("^GSPC", start="1970-02-01", end="2026-02-01", interval="1d")
sp500_monthly = sp500.resample("MS").first() 

usd_gbp = fred.get_series("EXUSUK", observation_start = "1977-12-01", frequency = "m")

cpi = fred.get_series("CPIAUCSL", observation_start = "1970-02-01") # M start 1947
gdp = fred.get_series("GDPC1", observation_start = "1970-02-01") # Q start Q1 1947
unemploy = fred.get_series("UNRATE", observation_start = "1970-02-01") # M start 1948
fed_rate = fred.get_series("FEDFUNDS", observation_start ="1970-02-01") # M start Jul 1954
T_yield = fred.get_series("GS10", observation_start = "1970-02-01") # 10-year treasury yield, M start Jan 1962
retail_sales = fred.get_series("RSAFS") # M starting 1992 # RETAILSMNSA start 1967, excludes food sales and is not seasonally adjusted
custom_spending = fred.get_series("PCE", observation_start = "1970-02-01") # M start 1959

[*********************100%***********************]  1 of 1 completed


In [38]:
sp500_monthly.head()

Price,Close,High,Low,Open,Volume
Ticker,^GSPC,^GSPC,^GSPC,^GSPC,^GSPC
Date,,,,,
1970-02-01,85.750000,86.760002,84.760002,85.019997,13440000
1970-03-01,89.709999,90.800003,88.919998,89.500000,12270000
1970-04-01,90.070000,90.620003,89.300003,89.629997,9810000
1970-05-01,81.440002,82.320000,80.269997,81.519997,8290000
1970-06-01,77.839996,78.400002,75.839996,76.550003,15020000


In [39]:
gdp_monthly = gdp.resample('MS').ffill()
gdp_monthly = gdp_monthly[2:]

sp500_data = sp500_monthly.drop(columns = [(  'High', '^GSPC'), (   'Low', '^GSPC'), (  'Open', '^GSPC')])
sp500_data.columns = sp500_data.columns.to_flat_index()

In [40]:
gdp_monthly

1970-03-01     5300.652
1970-04-01     5308.164
1970-05-01     5308.164
1970-06-01     5308.164
1970-07-01     5357.077
                ...    
2025-06-01    23770.976
2025-07-01    24026.834
2025-08-01    24026.834
2025-09-01    24026.834
2025-10-01    24065.956
Freq: MS, Length: 668, dtype: float64

In [41]:
print(sp500_data.columns)
sp500_df = sp500_data.rename(columns = {('Close', '^GSPC'):'S&P500 Close', ('Volume', '^GSPC'):'S&P500 Volume'})
sp500_df.head()

Index([('Close', '^GSPC'), ('Volume', '^GSPC')], dtype='object')


,S&P500 Close,S&P500 Volume
Date,,
1970-02-01,85.750000,13440000
1970-03-01,89.709999,12270000
1970-04-01,90.070000,9810000
1970-05-01,81.440002,8290000
1970-06-01,77.839996,15020000


In [42]:
econ_list = [cpi, gdp_monthly,unemploy, fed_rate, T_yield, retail_sales, custom_spending]
econ_df = pd.concat(econ_list, axis = 1)
merged_df = sp500_df.join(econ_df, how = 'outer')

In [43]:
df = merged_df.rename(columns = {0:'CPI', 1:'GDP', 2:'Unemployment Rate', 3:'Fed Rate', 4:'10 Year Treasury Yield', 5:'Retail Sales', 6:'Customer Spending'})

In [48]:
df['CPI'] = df['CPI'].interpolate(method = 'time')
df['Unemployment Rate'] = df['Unemployment Rate'].interpolate(method = 'time')
df.loc['2025-10-01':'2025-12-01', 'GDP'] = 24065.956
df.loc['1970-02-01', 'GDP'] = 5300.652

In [45]:
df = df[:-3]

In [49]:
df.head()

,S&P500 Close,S&P500 Volume,CPI,GDP,Unemployment Rate,Fed Rate,10 Year Treasury Yield,Retail Sales,Customer Spending
1970-02-01,85.750000,13440000.0,38.1,5300.652,4.2,8.98,7.24,NaN,634.0
1970-03-01,89.709999,12270000.0,38.3,5300.652,4.4,7.76,7.07,NaN,632.3
1970-04-01,90.070000,9810000.0,38.5,5308.164,4.6,8.10,7.39,NaN,636.0
1970-05-01,81.440002,8290000.0,38.6,5308.164,4.8,7.95,7.91,NaN,642.4
1970-06-01,77.839996,15020000.0,38.8,5308.164,4.9,7.61,7.84,NaN,646.3


In [50]:
df.to_csv("FIXED Economic Data.csv")